# LG Aimers Trackman-v2 CatBoost

2023에서 feature block을 선택하고, 선택된 A/I만 2024 holdout에서 확인하는 Colab 실행 노트북입니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone -b agent/tmv2-paper-features https://github.com/tswaincae1221/lg_aimers_experiment_lab.git
%cd /content/lg_aimers_experiment_lab
!pip -q install catboost==1.2.8 openpyxl


In [ ]:
import os
os.environ['TMV2_ROOT']='/content/tmv2_work'
os.environ['TMV2_OUT']='/content/tmv2_work/tmv2_experiment'
os.environ['TMV2_TRAIN']='/content/drive/MyDrive/aimers_data/train.csv'
os.environ['TMV2_TRACKMAN']='/content/drive/MyDrive/aimers_data/trackman_history.csv'
os.environ['TMV2_MAPPING']='/content/lg_aimers_experiment_lab/resources/pitcher_trackman_mapping.csv'
os.environ['TMV2_TASK_TYPE']='GPU'
os.environ['TMV2_DEVICES']='0'
print(os.environ['TMV2_ROOT'])


## 1. Hand-fixed Cat104 cache
`batter_hand 1→Left, 2→Right` normalization is applied before the Trackman context join.


In [ ]:
!python experiments/tmv2/prepare_tmv2_cache.py --repo-root /content/lg_aimers_experiment_lab --train "$TMV2_TRAIN" --trackman "$TMV2_TRACKMAN" --mapping "$TMV2_MAPPING" --work-dir "$TMV2_ROOT"


## 2. TM-v2 leakage-safe physical features


In [ ]:
!python experiments/tmv2/build_tmv2_features.py


## 3. 2023 selection screen


In [ ]:
labels=['A_baseline104','B_release_axis','C_plus_ellipse','D_plus_true_context','E_plus_fatigue','F_plus_movement','G_plus_mechanics_change','H_G_clean_old_tm','I_G_clean_plus_archetype']
import subprocess, os
for label in labels:
    print('\n###',label)
    subprocess.run(['python','experiments/tmv2/run_tmv2_screen.py','--label',label,'--valid','2023','--iterations','150'],check=True)


In [ ]:
import pandas as pd, glob, os
files=glob.glob(os.environ['TMV2_OUT']+'/score_*_2023.csv')
res=pd.concat([pd.read_csv(f) for f in files],ignore_index=True).sort_values('brier')
res[['label','feature_count','brier','auc','prediction_mean']]


## 4. 2024 final holdout
2023 winner was `I_G_clean_plus_archetype`; compare only A and I here.


In [ ]:
for label in ['A_baseline104','I_G_clean_plus_archetype']:
    subprocess.run(['python','experiments/tmv2/run_tmv2_screen.py','--label',label,'--valid','2024','--iterations','150'],check=True)


In [ ]:
files=glob.glob(os.environ['TMV2_OUT']+'/score_*_2024.csv')
pd.concat([pd.read_csv(f) for f in files],ignore_index=True).sort_values('brier')[['label','feature_count','brier','score','auc','bias']]


## 5. Native GPU challenger
After the screening result is reproduced, run the production comparison with native categorical handling. Select iteration on 2023 before treating the 2024 result as final.


In [ ]:
for label in ['A_baseline104','I_G_clean_plus_archetype']:
    subprocess.run(['python','experiments/tmv2/run_tmv2_native.py','--label',label,'--valid','2024','--iterations','216'],check=True)
